In [ ]:
!pip install transformers datasets evaluate sacrebleu accelerate -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset("dvgodoy/yoda_sentences")
dataset

In [ ]:
dataset["train"][0]

In [ ]:
dataset["train"].column_names

In [ ]:
dataset_split = dataset["train"].train_test_split(test_size=0.2, seed=42)

train_dataset = dataset_split["train"]
val_dataset = dataset_split["test"]

len(train_dataset), len(val_dataset)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google-t5/t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
source_column = "sentence"
target_column = "translation_extra"

max_input_length = 64
max_target_length = 64

def preprocess_function(examples):
    inputs = [
        "translate English to Yoda style: " + text
        for text in examples[source_column]
    ]

    targets = examples[target_column]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)
tokenized_train[0]

In [ ]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-yoda-style",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    predict_with_generate=True,
    logging_steps=10,
    save_total_limit=2,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

In [ ]:
def generate_yoda(sentence):
    input_text = "translate English to Yoda style: " + sentence

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
test_sentences = [
    "I am ready to study.",
    "You must finish your homework.",
    "The model learns from the data.",
    "This project is difficult but interesting.",
    "I need more coffee today.",
    "The student wrote a report about neural networks.",
    "We trained a model on a small dataset."
]

for sentence in test_sentences:
    print("Input:", sentence)
    print("Yoda:", generate_yoda(sentence))
    print()